In [ ]:
!pip install plotly

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px

from pathlib import Path

plt.style.use("ggplot")

In [ ]:
BASE_DIR = Path().resolve().parent

CLEANED = BASE_DIR/"data"/"cleaned"

In [ ]:
customers = pd.read_csv(CLEANED/"customers_cleaned.csv")

orders = pd.read_csv(CLEANED/"orders_cleaned.csv")

products = pd.read_csv(CLEANED/"products_cleaned.csv")

order_items = pd.read_csv(CLEANED/"order_items_cleaned.csv")

payments = pd.read_csv(CLEANED/"payments_cleaned.csv")

reviews = pd.read_csv(CLEANED/"reviews_cleaned.csv")

sellers = pd.read_csv(CLEANED/"sellers_cleaned.csv")

geo = pd.read_csv(CLEANED/"geolocation_cleaned.csv")

category = pd.read_csv(CLEANED/"category_translation_cleaned.csv")

In [ ]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Products": products,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Sellers": sellers,
    "Geolocation": geo,
    "Category": category
}

overview = pd.DataFrame({
    "Dataset":[k for k in datasets],
    "Rows":[v.shape[0] for v in datasets.values()],
    "Columns":[v.shape[1] for v in datasets.values()],
    "Missing Values":[v.isnull().sum().sum() for v in datasets.values()]
})

overview

# Dataset Overview

The Olist e-commerce dataset consists of multiple relational tables representing customers, orders, products, sellers, payments, reviews, and geolocation information.

The dataset was successfully cleaned and validated before analysis.

No major data quality issues remain that would impact business analysis.

In [ ]:
def quality_report(df):

    return pd.DataFrame({

        "Missing":df.isnull().sum(),

        "Unique":df.nunique(),

        "Datatype":df.dtypes

    })

In [ ]:
quality_report(customers)


In [ ]:
quality_report(orders)


In [ ]:
quality_report(products)

In [ ]:
quality_report(customers).to_csv(
    BASE_DIR/"reports"/"customers_quality.csv"
)

### Data Quality Assessment

All datasets were examined for missing values, duplicate records, unique identifiers, and data types.

The cleaning phase successfully removed major inconsistencies while preserving valid business records.

These datasets are suitable for exploratory analysis and downstream machine learning tasks.

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(order_items["price"],bins=60)

plt.title("Price Distribution")

plt.xlabel("Price")

plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(order_items["freight_value"],bins=60)

plt.title("Freight Distribution")

plt.show()

In [ ]:
reviews["review_score"].value_counts().sort_index().plot.bar()
plt.title("Review Score Distribution")
plt.show()

In [ ]:
payments["payment_type"].value_counts().plot.bar()
plt.title("Payment Methods")
plt.show()

In [ ]:
orders["order_status"].value_counts().plot.bar()
plt.title("Order Status")
plt.show()

In [ ]:
cat = products.merge(category, on="product_category_name")

top = cat["product_category_name_english"].value_counts().head(20)

top.plot.bar(figsize=(12,5))

In [ ]:
plt.hist(products["product_weight_g"], bins=50)
plt.title("Product Weight")
plt.show()

In [ ]:
plt.scatter(
    order_items["price"],
    order_items["freight_value"],
    alpha=0.3
)
plt.xlabel("Price")
plt.ylabel("Freight")

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Convert the text columns into proper Datetime objects
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])

# 2. Merge Orders + Reviews
# We merge on 'order_id' because it is the common link between both tables
merged = pd.merge(orders, reviews, on="order_id", how="inner")

# 3. Create the 'delivery_days' column
# We subtract the purchase date from the delivery date, then use .dt.days to extract the whole number
merged["delivery_days"] = (
    merged["order_delivered_customer_date"] - merged["order_purchase_timestamp"]
).dt.days

# 4. Create and display the Boxplot
fig = px.box(
    merged,
    x="review_score",
    y="delivery_days",
    title="Delivery Days vs. Review Score",
    labels={
        "review_score": "Review Score (1-5)",
        "delivery_days": "Total Days to Deliver"
    }
)

fig.show()

In [ ]:
payments.boxplot(
    column="payment_value",
    by="payment_type"
)

In [ ]:
merged.groupby(
"product_category_name_english"
)["price"].mean()

In [ ]:
pivot = merged.pivot_table(
values="price",
index="customer_state",
columns="product_category_name_english",
aggfunc="sum"
)

In [ ]:
plt.imshow(pivot)

In [ ]:
plt.boxplot(order_items["price"])

In [ ]:
plt.boxplot(order_items["freight_value"])

In [ ]:
plt.boxplot(products["product_weight_g"])

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(
orders["order_purchase_timestamp"]
)

In [ ]:
orders["month"] = orders["order_purchase_timestamp"].dt.month
orders["year"] = orders["order_purchase_timestamp"].dt.year
orders["weekday"] = orders["order_purchase_timestamp"].dt.day_name()
orders["hour"] = orders["order_purchase_timestamp"].dt.hour

In [ ]:
orders.groupby("month").size().plot()

In [ ]:
orders["weekday"].value_counts().plot.bar()

In [ ]:
orders["hour"].value_counts().sort_index().plot()

In [ ]:
state = merged.groupby(
"customer_state"
)["price"].sum()

state.plot.bar(figsize=(12,5))

In [ ]:
merged["customer_state"].value_counts().plot.bar()

In [ ]:
numeric = merged.select_dtypes(include=np.number)

In [ ]:
corr = numeric.corr()

In [ ]:
plt.figure(figsize=(12,8))
plt.imshow(corr)
plt.xticks(range(len(corr)), corr.columns, rotation=90)
plt.yticks(range(len(corr)), corr.columns)
plt.colorbar()
plt.show()

Section 9: Business Insights

Create a markdown section with 15–20 findings, for example:

Most products are priced below the average, indicating a strong budget-oriented marketplace.
Credit cards dominate payment methods.
A few sellers generate a significant share of total revenue.
Longer delivery times tend to correspond with lower review scores.
Freight costs generally increase with product price and size.
Certain product categories contribute disproportionately to revenue.
Some states account for a much larger share of orders than others.
Product prices and freight charges exhibit right-skewed distributions.
Most orders are successfully delivered; cancellations are relatively rare.
Customer reviews are concentrated at higher ratings, suggesting overall satisfaction.
Product weights vary widely, reflecting a diverse catalog.
Seller distribution is uneven across states.
Revenue shows seasonal fluctuations over time.
Peak ordering hours indicate customer shopping behavior.
Outliers in price and freight are likely valid high-value transactions rather than data errors.

In [ ]:
plt.savefig(
OUTPUT_DIR/"figures"/"price_distribution.png",
dpi=300,
bbox_inches="tight"
)

In [ ]:
plt.savefig(
OUTPUT_DIR/"figures"/"price_distribution.png",
dpi=300,
bbox_inches="tight"
)